# Домашнее задание: Спекулятивное декодирование и архитектура Qwen

**Курс:** NLP-2

## Описание задания

В этом домашнем задании мы не будем заниматься обучением моделей с нуля. Вместо этого мы сфокусируемся на **Inference Engineering** — области, которая находится на стыке разработки и исследований и занимается оптимизацией и ускорением работы уже обученных больших языковых моделей (LLM).

Мы реализуем с нуля метод **Speculative Decoding** — один из самых популярных алгоритмических подходов, позволяющий ускорить генерацию текста в 1.5-2.5 раза практически без потери качества. Этот метод используется для оптимизации инференса таких моделей, как Llama, Mixtral и других.

### План работы:
1. **Подготовка окружения**: Настроим среду и загрузим модели.
2. **Реализация Draft модели**: Мы вручную, блок за блоком, соберем архитектуру модели `Qwen2.5-0.5B` (`NanoQwen`). Это позволит нам досконально понять устройство современных LLM, включая `RMSNorm`, `RoPE` и `SwiGLU`.
3. **Реализация цикла спекуляции**: Напишем основной алгоритм, в котором Draft модель быстро генерирует черновик, а Target модель его верифицирует.
4. **Бенчмарк**: Проведем замеры и оценим реальное ускорение, которое дает наш метод.

## Шаг 1: Настройка окружения

!pip install -q transformers accelerate safetensors sentencepiece

!pip install -q bitsandbytes

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    BitsAndBytesConfig
)
from tqdm import tqdm
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import time
import gc
import json
import pickle
import numpy as np
import pandas as pd
import bitsandbytes as bnb
import random
import copy

# Фиксируем seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


### Загрузка моделей

В качестве целевой модели (`Target`) мы будем использовать `Qwen/Qwen2.5-1.5B-Instruct`. В качестве черновой модели (`Draft`) — `Qwen/Qwen2.5-0.5B-Instruct`. Мы загрузим `Target` с помощью стандартного класса `AutoModelForCausalLM` из `transformers`, а вот `Draft` модель соберем вручную.

In [2]:
TARGET_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DRAFT_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID)

print("Загрузка Target модели (через AutoModelForCausalLM)...")
# Используем torch_dtype=torch.float16 для экономии VRAM и attn_implementation="sdpa" для использования Flash Attention
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
target_model.eval()
print("Модель Target загружена.")

Загрузка токенизатора...
Загрузка Target модели (через AutoModelForCausalLM)...
Модель Target загружена.


## Шаг 2: Собираем Draft модель "NanoQwen" (4 балла)

На этом шаге мы не будем использовать `AutoModel`, а соберем модель самостоятельно из отдельных модулей. Это ключевая часть задания, которая поможет понять, как современные трансформеры устроены "под капотом".

Сначала загрузим только конфигурацию `Draft` модели, из которой мы будем брать все параметры архитектуры (размер скрытого слоя, количество голов и т.д.).

In [3]:
draft_config = AutoConfig.from_pretrained(DRAFT_ID)
print(f"Конфигурация Draft модели:")
print(f"  - Размер скрытого слоя (hidden_size): {draft_config.hidden_size}")
print(f"  - Количество слоев (num_hidden_layers): {draft_config.num_hidden_layers}")
print(f"  - Количество голов внимания (num_attention_heads): {draft_config.num_attention_heads}")

Конфигурация Draft модели:
  - Размер скрытого слоя (hidden_size): 896
  - Количество слоев (num_hidden_layers): 24
  - Количество голов внимания (num_attention_heads): 14


### Задание 2.1: RMSNorm

Современные модели, такие как Llama и Qwen, используют **Root Mean Square Layer Normalization (RMSNorm)** вместо классического `LayerNorm`. `RMSNorm` проще и вычислительно эффективнее, так как оперирует только масштабированием на основе среднеквадратичного значения и не использует дополнительный сдвиг (bias).

Формула `RMSNorm` для вектора активаций $\mathbf{x}$:
$$ \text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2} $$
$$ \text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x}) + \epsilon} \cdot \mathbf{w} $$
где $\mathbf{w}$ — обучаемый весовой вектор (гейт), а $\epsilon$ — малая константа для численной стабильности.

**Важно**: При вычислениях в `float16`, промежуточный расчет `variance` (степень `pow(2)`) может привести к переполнению. Поэтому стандартная практика — временно повышать тип данных до `float32` для этого расчета.

**Задание**: Реализуйте `forward` для `QwenRMSNorm`.

**Полезные ссылки**:
- [Root Mean Square Layer Normalization (Zhang and Sennrich, 2019)](https://arxiv.org/abs/1910.07467)

In [4]:
class QwenRMSNorm(nn.Module):
    """
    Реализация Root Mean Square Layer Normalization.

    Аргументы:
        hidden_size (int): Размер скрытого слоя.
        eps (float): Малая константа для предотвращения деления на ноль.
    """
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        # --- НАЧАЛО ВАШЕГО КОДА ---
        # hidden_states: [b, seq_len, hidden_size]
        # 1. Запомните исходный тип данных (например, float16).
        orig_dtype = hidden_states.dtype
        # 2. Переведите hidden_states в float32 для стабильных вычислений.
        hidden_states = hidden_states.to(torch.float32)  # [b, seq_len, hidden_size]
        # 3. Вычислите variance: среднее от квадратов элементов по последней оси.
        #    Не забудьте `keepdim=True`: [b, seq_len, 1].
        variance = hidden_states.pow(2).mean(dim=-1, keepdim=True)
        # 4. Нормализуйте hidden_states (torch.rsqrt в помощь).
        # [b, seq_len, 1] broadcasting -> [b, seq_len, hidden_size] нормировка по каждому токену
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        # 5. Умножьте на обучаемый вес и верните результат в исходном типе данных.
        # [b, seq_len, hidden_size] x [b, seq_len, hidden_size] -> [b, seq_len, hidden_size]
        output = hidden_states * self.weight  # broadcasting, веса применяем к каждому токену
        # --- КОНЕЦ ВАШЕГО КОДА ---
        return output.to(orig_dtype)

In [5]:
print("--- Запуск тестов для RMSNorm ---")
try:
    hidden_size = 128
    norm_layer = QwenRMSNorm(hidden_size).to(device).half()

    # Тест 1: Проверка размерности
    dummy_input = torch.randn(2, 10, hidden_size, device=device).half()
    output = norm_layer(dummy_input)
    assert output.shape == dummy_input.shape, f"Ошибка размерности: ожидалось {dummy_input.shape}, получено {output.shape}"
    print("✅ [1/3] Тест на размерность пройден.")

    # Тест 2: Проверка типа данных
    assert output.dtype == torch.float16, f"Ошибка типа данных: ожидалось float16, получено {output.dtype}"
    print("✅ [2/3] Тест на тип данных пройден.")

    # Тест 3: Проверка на NaN
    assert not torch.isnan(output).any(), "В выходе RMSNorm обнаружены NaN значения."
    print("✅ [3/3] Тест на NaN пройден.")

    print("\n🎉 Все тесты для RMSNorm пройдены!")

except Exception as e:
    print(f"❌ Тест RMSNorm провален: {e}")

--- Запуск тестов для RMSNorm ---
✅ [1/3] Тест на размерность пройден.
✅ [2/3] Тест на тип данных пройден.
✅ [3/3] Тест на NaN пройден.

🎉 Все тесты для RMSNorm пройдены!


### Задание 2.2: Rotary Positional Embeddings (RoPE)

`RoPE` — это элегантный способ внедрения позиционной информации, который вместо добавления векторов (как в `sin/cos embeddings`) "вращает" векторы запросов (`Query`) и ключей (`Key`) на угол, зависящий от их позиции.

#### Два подхода к реализации
Существует два эквивалентных способа реализации этого вращения.

1. **Разбиение на пары (Pairwise Rotation)**. Этот метод мы рассматривали на лекции. Вектор признаков $\mathbf{x} = (x_1, x_2, \dots, x_d)$ рассматривается как набор двумерных векторов $(x_{2i-1}, x_{2i})$. Каждый такой вектор вращается в 2D-плоскости:
$$
\begin{pmatrix} x'_{2i-1} \\ x'_{2i} \end{pmatrix} =
\begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}
\begin{pmatrix} x_{2i-1} \\ x_{2i} \end{pmatrix}
$$
где $m$ — позиция токена, а $\theta_i$ — частота вращения.

2. **Вращение через половины (`rotate_half`)**. Этот метод используется в реализациях Llama и Qwen, и именно его мы будем использовать. Здесь вектор $\mathbf{x}$ делится на две половины: $\mathbf{x}_1 = (x_1, \dots, x_{d/2})$ и $\mathbf{x}_2 = (x_{d/2+1}, \dots, x_d)$. Вращение реализуется следующим образом:
$$ \mathbf{x}'_m = \mathbf{x}_m \cos(m\theta) + \text{rotate\_half}(\mathbf{x}_m) \sin(m\theta) $$
где операция `rotate_half` преобразует вектор $\mathbf{x} = (\mathbf{x}_1, \mathbf{x}_2)$ в $(-\mathbf{x}_2, \mathbf{x}_1)$. Этот подход легче векторизуется и более эффективен в `torch`.

**Задание**: Реализуйте функцию `apply_rope`, следуя второму подходу.

**Полезные ссылки**:
- [RoFormer: Enhanced Transformer with Rotary Position Embedding (Su et al., 2021)](https://arxiv.org/abs/2104.09864)
- [Подробное объяснение RoPE в блоге EleutherAI](https://blog.eleuther.ai/rotary-embeddings/)

In [6]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 1000000.0):
    """
    Предварительно вычисляет частоты для RoPE в комплексном виде (cos + i*sin).

    Эта функция готовит таблицы синусов и косинусов для всех возможных позиций.
    """
    # [dim//2], где каждая частота ω_i=1/θ**(i/d)
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    # [end]  массив номеров позиций токенов (0, 1, ..., end-1)
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    # матрицу всех комбинаций позиций и частот [end, dim//2]
    freqs = torch.outer(t, freqs)
    # В реализации Llama/Qwen частоты дублируются для обеих половин
    # Мы вернем косинусы и синусы отдельно для удобства
    # [end, dim//2] -> [end, dim]
    freqs_cat = torch.cat((freqs, freqs), dim=-1)
    return torch.cos(freqs_cat), torch.sin(freqs_cat)

def rotate_half(x):
    """Вращает половину скрытых измерений входа."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    """
    Применяет RoPE (rotary positional embeddings) к векторам q и k через rotate_half.
    q, k: [batch, seq_len, num_heads, head_dim]
    cos, sin: [seq_len, head_dim]
    Выход: q_embed, k_embed того же формата и типа (float16 или bfloat16)
    """
    # --- НАЧАЛО ВАШЕГО КОДА ---
    # 1. Измените размерность cos и sin для бродкастинга с (q, k).
    #    [seq_len, head_dim] -> [1, seq_len, 1, head_dim].
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    # 2. Примените формулу вращения к q и k, используя rotate_half.
    #    Операции с float32 (cos/sin) приведут к результату float32.
    #    x' = x * cos + rotate_half(x) * sin
    # [batch, seq_len, num_heads, head_dim] * [1, seq_len, 1, head_dim]: broadcasting
    # -> [batch, seq_len, num_heads, head_dim]
    q_rot = rotate_half(q)
    k_rot = rotate_half(k)
    q_embed = q * cos + q_rot * sin
    k_embed = k * cos + k_rot * sin
    # 3. Верните q_embed и k_embed Приводим тип обратно к типу входа (например, float16).
    q_embed = q_embed.to(q.dtype)
    k_embed = k_embed.to(k.dtype)
    #    ВАЖНО: Явно приводим к типу q.dtype (float16), иначе упадет assert или следующий слой
    # --- КОНЕЦ ВАШЕГО КОДА ---
    return q_embed, k_embed

In [7]:
print("--- Запуск тестов для RoPE ---")
try:
    head_dim = 64
    seq_len = 10
    batch_size = 2
    num_heads = 4

    xq = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device).half()
    xk = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device).half()
    cos, sin = precompute_freqs_cis(head_dim, seq_len * 2)
    cos, sin = cos.to(device), sin.to(device)

    # Вызываем функцию
    xq_rot, xk_rot = apply_rope(xq, xk, cos[:seq_len], sin[:seq_len])

    # Тест 1: Проверка размерности
    assert xq_rot.shape == xq.shape, f"Ошибка размерности Query: ожидалось {xq.shape}, получено {xq_rot.shape}"
    assert xk_rot.shape == xk.shape, f"Ошибка размерности Key: ожидалось {xk.shape}, получено {xk_rot.shape}"
    print("✅ [1/2] Тест на размерность пройден.")

    # Тест 2: Проверка типа данных
    assert xq_rot.dtype == xq.dtype, f"Ошибка типа данных Query: ожидалось {xq.dtype}, получено {xq_rot.dtype}"
    assert xk_rot.dtype == xk.dtype, f"Ошибка типа данных Key: ожидалось {xk.dtype}, получено {xk_rot.dtype}"
    print("✅ [2/2] Тест на тип данных пройден.")

    print("\n🎉 Все тесты для RoPE пройдены!")
except Exception as e:
    print(f"❌ Тест RoPE провален: {e}")

--- Запуск тестов для RoPE ---
✅ [1/2] Тест на размерность пройден.
✅ [2/2] Тест на тип данных пройден.

🎉 Все тесты для RoPE пройдены!


### Задание 2.3: SwiGLU MLP

Вместо стандартного `FeedForward` блока с одной `ReLU` активацией, современные модели используют `Gated Linear Units (GLU)` и их варианты. В Qwen/Llama используется **SwiGLU**.

Идея состоит в том, чтобы использовать гейт (шлюз) для управления информационным потоком. Входной вектор `x` проецируется двумя разными линейными слоями (`up` и `gate`). Результат `gate` проекции проходит через активацию `SiLU` (также известную как Swish), а затем поэлементно умножается на результат `up` проекции. Это позволяет сети динамически решать, какая информация должна пройти дальше.

Формула `SwiGLU`:
$$ \text{SwiGLU}(x, W_{up}, W_{gate}, W_{down}) = (\text{SiLU}(x W_{gate}) \otimes (x W_{up})) W_{down} $$
где $\otimes$ — поэлементное умножение, а `SiLU` (Swish activation) определяется как:
$$ \text{SiLU}(x) = x \cdot \sigma(x) $$
где $\sigma$ — это сигмоида.

**Задание**: Реализуйте `forward` для `QwenMLP`.

**Полезные ссылки**:
- [GLU Variants Improve Transformer (Shazeer, 2020)](https://arxiv.org/abs/2002.05202)

In [8]:
class QwenMLP(nn.Module):
    """
    Реализация SwiGLU Feed-Forward сети.
    """
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        # x: [batch, seq_len, hidden_size]
        # --- НАЧАЛО ВАШЕГО КОДА ---
        pass
        # 1. Примените gate_proj и up_proj к входу x.
        gate = self.gate_proj(x)  # [batch, seq_len, intermediate_size]
        up = self.up_proj(x)      # [batch, seq_len, intermediate_size]
        # 2. Примените активацию SiLU (F.silu) к выходу gate_proj.
        gate_act = F.silu(gate)  # [batch, seq_len, intermediate_size]
        # 3. Поэлементно перемножьте результат шага 2 и выход up_proj.
        hidden = gate_act * up   # [batch, seq_len, intermediate_size]
        # 4. Пропустите результат через down_proj и верните его.

        # --- КОНЕЦ ВАШЕГО КОДА ---
        return self.down_proj(hidden)  # [batch, seq_len, hidden_size]

In [9]:
print("--- Запуск тестов для SwiGLU MLP ---")
try:
    # Создаем mock-конфиг для теста
    class MockConfig:
        hidden_size = 128
        intermediate_size = 384

    config = MockConfig()
    mlp = QwenMLP(config).to(device).half()

    x = torch.randn(2, 10, config.hidden_size, device=device).half()
    output = mlp(x)

    # Тест 1: Проверка размерности
    assert output.shape == x.shape, f"Ошибка размерности: ожидалось {x.shape}, получено {output.shape}"
    print("✅ [1/2] Тест на размерность пройден.")

    # Тест 2: Проверка на NaN
    assert not torch.isnan(output).any(), "В выходе MLP обнаружены NaN значения."
    print("✅ [2/2] Тест на NaN пройден.")

    print("\n🎉 Все тесты для SwiGLU MLP пройдены!")
except Exception as e:
    print(f"❌ Тест SwiGLU MLP провален: {e}")

--- Запуск тестов для SwiGLU MLP ---
✅ [1/2] Тест на размерность пройден.
✅ [2/2] Тест на NaN пройден.

🎉 Все тесты для SwiGLU MLP пройдены!


### Задание 2.4: Сборка итоговой модели NanoQwen

Теперь, когда у нас есть все строительные блоки, мы можем собрать из них полноценный слой трансформера (`NanoQwenBlock`) и саму модель (`NanoQwen`).

Вам необходимо дополнить класс `NanoQwenBlock`, правильно соединив все модули. Обратите внимание на порядок операций в трансформерах семейства Llama/Qwen:
1. **Pre-normalization** перед Self-Attention.
2. **Residual Connection** после Self-Attention.
3. **Pre-normalization** перед MLP.
4. **Residual Connection** после MLP.

In [10]:
class QwenAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=True)
        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)
        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

    def forward(self, hidden_states, cos, sin):
        # hidden_states: [bsz, q_len, hidden_size]
        bsz, q_len, _ = hidden_states.size()

        # q: [bsz, q_len, num_heads, head_dim]
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim)
        # k: [bsz, q_len, num_key_value_heads, head_dim]
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim)
        # v: [bsz, q_len, num_key_value_heads, head_dim]
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim)
        # q: [bsz, q_len, num_heads, head_dim], k: [bsz, q_len, num_key_value_heads, head_dim]
        q, k = apply_rope(q, k, cos, sin)

        # Grouped Query Attention (GQA)
        if self.num_key_value_heads != self.num_heads:
            # иногда Multi-Query Attention -> [bsz, q_len, num_heads, head_dim]
            k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=2)
            v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=2)
            # одна голова k/v "обслуживает" несколько голов q

        q = q.transpose(1, 2)  # [bsz, num_heads, q_len, head_dim]
        k = k.transpose(1, 2)  # [bsz, num_heads, q_len, head_dim]
        v = v.transpose(1, 2)  # [bsz, num_heads, q_len, head_dim]

        # Используем встроенную реализацию Flash Attention
        # [bsz, num_heads, q_len, head_dim]
        output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        # output после .transpose: [bsz, q_len, num_heads, head_dim]
        # после .view: [bsz, q_len, num_heads * head_dim] = [bsz, q_len, hidden_size]
        output = output.transpose(1, 2).contiguous().view(bsz, q_len, -1)
        return self.o_proj(output)  # [bsz, q_len, hidden_size]

class NanoQwenBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = QwenAttention(config)
        self.mlp = QwenMLP(config)
        self.input_layernorm = QwenRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = QwenRMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, cos, sin):
        # hidden_states: [batch, seq_len, hidden_size]
        # --- НАЧАЛО ВАШЕГО КОДА ---
        # 1.1 Pre-normalization
        normed_attn = self.input_layernorm(hidden_states)  #  [batch, seq_len, hidden_size]
        # 1.2 Self-Attention
        attn_out = self.self_attn(normed_attn, cos, sin)  # [batch, seq_len, hidden_size]
        # 1.3 Residual connection после attention
        hidden_states = hidden_states + attn_out  # [batch, seq_len, hidden_size]

        # 2.1 Pre-normalization перед MLP
        normed_mlp = self.post_attention_layernorm(hidden_states)  # [batch, seq_len, hidden_size]
        # 2.2 MLP
        mlp_out = self.mlp(normed_mlp)  # [batch, seq_len, hidden_size]
        # 2.3 Residual connection после MLP
        hidden_states = hidden_states + mlp_out  # [batch, seq_len, hidden_size]
        # --- КОНЕЦ ВАШЕГО КОДА ---

        return hidden_states

class NanoQwen(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([NanoQwenBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = QwenRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Предвычисляем RoPE
        self.cos, self.sin = precompute_freqs_cis(
            config.hidden_size // config.num_attention_heads,
            config.max_position_embeddings * 2
        )
        self.cos = self.cos.to(device)
        self.sin = self.sin.to(device)

    def forward(self, input_ids):
        x = self.embed_tokens(input_ids)
        seq_len = x.shape[1]

        # Выбираем нужный срез из таблицы RoPE
        cos_t = self.cos[:seq_len]
        sin_t = self.sin[:seq_len]

        for layer in self.layers:
            x = layer(x, cos_t, sin_t)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

### Загрузка весов

Теперь нам нужно загрузить веса из официального репозитория в нашу самописную модель. Мы уже реализовали функцию `load_weights_manual`, которая делает это и корректно обрабатывает различия в именовании слоев.

In [11]:
def load_weights_manual(model, model_id):
    """
    Загружает веса из файла .safetensors в ручном режиме,
    корректируя имена ключей.
    """
    print("Скачивание весов...")
    file_path = hf_hub_download(repo_id=model_id, filename="model.safetensors")

    print("Загрузка файла safetensors...")
    state_dict = load_file(file_path, device="cpu") # Грузим на CPU, чтобы не занимать VRAM

    new_state_dict = {}
    for key, tensor in state_dict.items():
        new_key = key

        # Если ключ начинается с "model.", удаляем этот префикс
        if key.startswith("model."):
            new_key = key[len("model."):]

        new_state_dict[new_key] = tensor

    # В некоторых моделях lm_head и embed_tokens используют общие веса (tied weights).
    # Если lm_head нет, копируем его из embed_tokens.
    if "lm_head.weight" not in new_state_dict and "embed_tokens.weight" in new_state_dict:
        print("Связывание весов: lm_head.weight <- embed_tokens.weight")
        new_state_dict["lm_head.weight"] = new_state_dict["embed_tokens.weight"]

    print("Загрузка весов в модель...")
    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)

    if not missing and not unexpected:
        print("✅ Веса успешно загружены!")
    else:
        print(f"❌ Ошибка при загрузке весов:")
        if missing:
            print(f"  Не найдены ключи в state_dict: {missing[:5]}")
        if unexpected:
            print(f"  Лишние ключи в state_dict: {unexpected[:5]}")

    model.to(device).to(torch.float16)
    return model, missing, unexpected

In [12]:
# Инициализация и загрузка Draft модели
draft_model = NanoQwen(draft_config)
draft_model, missing_keys, unexpected_keys = load_weights_manual(draft_model, DRAFT_ID)
draft_model.eval()

# Тест
print("\n--- Запуск теста для загрузки весов ---")
try:
    assert not missing_keys, f"Найдены недостающие ключи: {missing_keys[:5]}"
    assert not unexpected_keys, f"Найдены лишние ключи: {unexpected_keys[:5]}"
    print("🎉 Тест на загрузку весов пройден!")
except Exception as e:
    print(f"❌ Тест провален: {e}")

Скачивание весов...
Загрузка файла safetensors...
Связывание весов: lm_head.weight <- embed_tokens.weight
Загрузка весов в модель...
✅ Веса успешно загружены!

--- Запуск теста для загрузки весов ---
🎉 Тест на загрузку весов пройден!


### Проверка работоспособности

Давайте убедимся, что наша самописная модель генерирует осмысленный текст. Мы используем простой цикл жадной генерации (`greedy search`).

In [13]:
def generate_simple(model, text, max_new=10):
    """Простая функция для жадной генерации."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs.input_ids

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        for _ in range(max_new):
            logits = model(input_ids)
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who are you?"}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("--- Ответ от NanoQwen: ---")
print(generate_simple(draft_model, text, max_new=20))

--- Ответ от NanoQwen: ---
system
You are a helpful assistant.
user
Who are you?
assistant
I am a large language model created by Alibaba Cloud. I am called Qwen.


## Шаг 3: Алгоритм спекулятивного декодирования (3 балла)

Теперь самая важная часть. Мы реализуем **Greedy Speculative Decoding**.

**Алгоритм**:
1. **Черновик (Draft)**: Генерируем `K` токенов с помощью быстрой маленькой модели (`Draft`).
2. **Верификация (Verify)**: Прогоняем всю последовательность (префикс + `K` токенов) через медленную большую модель (`Target`) **за один `forward` вызов**.
3. **Проверка (Accept/Reject)**: Сравниваем токены, предсказанные `Target` моделью, с токенами, сгенерированными `Draft` моделью.
   - Находим первый индекс `i`, где предсказания не совпали.
   - Все `i` совпавших токенов считаются "принятыми" и добавляются к результату.
   - В качестве следующего токена мы берем "правильный" токен от `Target` модели на позиции `i`.
   - Все остальные токены из черновика отбрасываются.
4. Повторяем цикл.

**Задание**: Реализуйте логику проверки и принятия токенов.

**Полезные ссылки**:
- [Fast Inference from Transformers via Speculative Decoding (Leviathan et al., 2022)](https://arxiv.org/abs/2211.17192)

In [13]:
def speculative_sampling(prefix_text, max_new_tokens, target_model, draft_model, tokenizer, K=5):
    """
    Реализует цикл спекулятивного декодирования.
    """
    # Форматируем промпт
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prefix_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

    generated_ids = input_ids.clone()
    finished_len = input_ids.shape[1] + max_new_tokens

    stats = {"target_calls": 0, "total_accepted": 0, "total_drafted": 0}

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        while generated_ids.shape[1] < finished_len:
            prefix_len = generated_ids.shape[1]

            # 1. DRAFT
            draft_ids = generated_ids
            for _ in range(K):
                outputs = draft_model(draft_ids)
                next_token = torch.argmax(outputs[:, -1, :], dim=-1, keepdim=True)
                draft_ids = torch.cat([draft_ids, next_token], dim=1)
                if next_token.item() == tokenizer.eos_token_id:
                    break

            drafted_tokens = draft_ids[0, prefix_len:]
            if not len(drafted_tokens): break # Если ничего не сгенерировали
            stats["total_drafted"] += len(drafted_tokens)

            # 2. VERIFY
            target_outputs = target_model(draft_ids)
            stats["target_calls"] += 1
            target_logits = target_outputs.logits
            # Нас интересуют предсказания Target модели для токенов, которые сгенерировал Draft
            relevant_logits = target_logits[0, prefix_len-1 : -1]
            target_preds = torch.argmax(relevant_logits, dim=-1)

            # 3. ACCEPT/REJECT Logic
            n_accepted = 0

            # --- НАЧАЛО ВАШЕГО КОДА ---
            # Проитерируйтесь по `drafted_tokens` и `target_preds`.
                # Увеличивайте `n_accepted`, пока токены совпадают.
                    # Прервите цикл, как только найдете первое несовпадение.
            # Обе последовательности одинаковой длины
            for i in range(len(drafted_tokens)):
                if drafted_tokens[i] == target_preds[i]:
                    n_accepted += 1
                else:
                    break  # Первое несовпадение
            # --- КОНЕЦ ВАШЕГО КОДА ---

            stats["total_accepted"] += n_accepted

            # Принимаем все совпавшие токены
            accepted_ids = drafted_tokens[:n_accepted]
            generated_ids = torch.cat([generated_ids, accepted_ids.unsqueeze(0)], dim=1)

            # Если достигли лимита, выходим
            if generated_ids.shape[1] >= finished_len:
                break

            # Проверка EOS: если последний принятый токен - EOS, останавливаемся
            if n_accepted > 0 and accepted_ids[-1].item() == tokenizer.eos_token_id:
                break

            # Добавляем один "исправленный" токен от Target модели
            if n_accepted < len(target_preds):
                correct_token = target_preds[n_accepted].view(1, 1)
            else: # Если все совпало, берем следующий токен от Target модели
                last_logits = target_logits[0, -1, :]
                correct_token = torch.argmax(last_logits).view(1, 1)

            generated_ids = torch.cat([generated_ids, correct_token], dim=1)

            if correct_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True), stats

In [15]:
print("--- Запуск теста для Speculative Decoding ---")
try:
    # Создаем "игрушечные" модели для теста
    class MockModel(nn.Module):
        def __init__(self, vocab_size, response_sequence):
            super().__init__()
            self.vocab_size = vocab_size
            self.response = response_sequence
            self.call_idx = 0
        def forward(self, input_ids):
            if input_ids is None: return None # property hack fix
            batch, seq_len = input_ids.shape
            next_token_idx = min(self.call_idx, len(self.response) -1)
            next_token = self.response[next_token_idx]
            self.call_idx += 1
            logits = torch.full((batch, seq_len, self.vocab_size), -100.0, device=device)
            logits[:, -1, next_token] = 100.0
            return logits

    mock_draft = MockModel(100, [10, 20, 30, 40, 50])
    mock_target = MockModel(100, [10, 20, 99, 40, 50])
    # Хамский хак для property, но в тесте мы вызываем forward напрямую через target(draft_ids)

    def mock_speculative_sampling(target, draft, K=5):
        generated_ids = torch.tensor([[1, 2]], device=device)
        prefix_len = generated_ids.shape[1]
        draft_ids = torch.tensor([[1, 2, 10, 20, 30, 40, 50]], device=device)

        drafted_tokens = draft_ids[0, prefix_len:]
        print(f"Drafted tokens: {drafted_tokens.tolist()}")

        # Эмулируем вызов target модели
        # Важно: MockModel в нашем тесте очень тупая, она просто возвращает следующий токен из списка response
        # независимо от входа. Нам нужно вызвать ее для каждого токена последовательности,
        # чтобы собрать "правильные" предикты.

        # Перепишем логику сбора предиктов для MockModel, чтобы она работала как авторегрессия
        target_preds_list = []

        # Мы хотим проверить предикты для:
        # input=[1, 2] -> предсказание (должно быть 10)
        # input=[... 10] -> предсказание (должно быть 20)
        # input=[... 20] -> предсказание (должно быть 99)

        # Сбросим состояние
        target.call_idx = 0

        # В реальной жизни мы делаем один forward pass.
        # Но наша MockModel возвращает только последний токен.
        # Поэтому для теста мы просто возьмем response_sequence учителя.
        # Это упрощение теста, но оно валидирует логику accept/reject.

        teacher_sequence = target.response # [10, 20, 99, 40, 50]
        target_preds = torch.tensor(teacher_sequence, device=device)

        print(f"Target preds (ideal): {target_preds.tolist()}")

        n_accepted = 0
        # --- КОПИЯ ВАШЕЙ ЛОГИКИ ИЗ speculative_sampling ---
        for i in range(len(drafted_tokens)):
            if drafted_tokens[i] == target_preds[i]:
                n_accepted += 1
            else:
                break  # Первое несовпадение
        # raise NotImplementedError("КОПИЯ ВАШЕЙ ЛОГИКИ ИЗ speculative_sampling")
        # --------------------------------------------------
        return n_accepted

    n_accepted = mock_speculative_sampling(mock_target, mock_draft)
    print(f"Accepted: {n_accepted}")

    assert n_accepted == 2, f"Ошибка в логике Accept/Reject: ожидалось 2 принятых токена, получено {n_accepted}"
    print("🎉 Тест для логики Accept/Reject пройден!")

except Exception as e:
    print(f"❌ Тест провален: {e}")


--- Запуск теста для Speculative Decoding ---
Drafted tokens: [10, 20, 30, 40, 50]
Target preds (ideal): [10, 20, 99, 40, 50]
Accepted: 2
🎉 Тест для логики Accept/Reject пройден!


## Шаг 4: Бенчмарк

Чтобы увидеть реальный выигрыш от спекулятивного декодирования, нам нужно симулировать ситуацию, когда `Target` модель работает значительно медленнее, чем `Draft`. В реальной жизни так и происходит: `Draft` может быть модель на 0.5B параметров, а `Target` — на 70B.

Мы создадим обертку `HeavyTarget`, которая будет искусственно добавлять задержку перед каждым вызовом `forward`, имитируя медленный инференс большой модели. Затем мы сравним время генерации стандартным (авторегрессионным) способом и с помощью нашего спекулятивного алгоритма.Ожидается, что спекулятивное декодирование покажет значительное ускорение (speedup ~ 1.5x+).

Однако, обратите внимание, что для действительно эффективной работы на длинных контекстах нам необходим KV-cache, чтобы заново не пересчитывать уже выполненные вычисления. Именно так speculative decoding реализован в популярных фреймворках.

In [14]:
class HeavyTarget:
    def __init__(self, model, delay=0.05):
        self.model = model
        self.delay = delay
    def __call__(self, *args, **kwargs):
        time.sleep(self.delay)
        return self.model(*args, **kwargs)
    @property
    def config(self): return self.model.config

def autoregressive(model, text, max_new=50):
    ids = tokenizer(text, return_tensors="pt").to(device).input_ids
    start = time.time()
    cnt = 0
    with torch.no_grad():
        for _ in range(max_new):
            out = model(ids)
            tok = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            ids = torch.cat([ids, tok], dim=1)
            cnt += 1
            if tok.item() == tokenizer.eos_token_id: break
    return cnt, time.time() - start

In [17]:
prompts = [
    "Write a Python function to calculate Fibonacci numbers.",
    "The capital of France is",
    "Explain the theory of relativity in simple terms."
]
heavy_target = HeavyTarget(target_model, 0.04)

results = []
print("Running Benchmark...")
for p in prompts:
    # Std
    s_tok, s_time = autoregressive(heavy_target, p)
    s_speed = s_tok / s_time

    # Spec
    start = time.time()
    _, stats = speculative_sampling(p, 50, heavy_target, draft_model, tokenizer)
    spec_time = time.time() - start
    spec_tok = stats['total_accepted'] + stats['target_calls']
    spec_speed = spec_tok / spec_time

    results.append({
        "Prompt": p[:20],
        "Std (t/s)": s_speed,
        "Spec (t/s)": spec_speed,
        "Speedup": spec_speed / s_speed
    })

print(pd.DataFrame(results).to_string(float_format="{:.2f}".format))

Running Benchmark...
                 Prompt  Std (t/s)  Spec (t/s)  Speedup
0  Write a Python funct      12.95       32.17     2.48
1  The capital of Franc      13.16       24.93     1.89
2  Explain the theory o      13.16       21.48     1.63


## Что дальше?

Чтобы получить максимальный балл за ДЗ, попробуйте реализовать одну из следующих идей:

### 1. Quantized Draft Model (0.5 балла)
Мы ускорили модель алгоритмически. А давайте теперь ускорим Draft модель аппаратно! Попробуйте квантовать `NanoQwen` в 4-бит (используя библиотеки `bitsandbytes` или `GPTQ`) и посмотрите, как изменится время генерации драфтов и итоговое ускорение (Speedup).

In [18]:
def create_quantized_draft_from_existing(draft_model):
    """
    Создает квантованную копию уже загруженной Draft модели
    """
    print("Создание квантованной копии Draft модели...")

    # Копируем модель (только структуру)
    draft_config = draft_model.config
    quantized_draft = NanoQwen(draft_config)

    # Копируем веса из оригинальной модели
    quantized_draft.load_state_dict(draft_model.state_dict())

    # Квантуем линейные слои
    def quantize_linear_layers(module):
        for name, child in module.named_children():
            if isinstance(child, nn.Linear):
                # Создаем квантованный аналог
                quantized_linear = bnb.nn.Linear4bit(
                    child.in_features,
                    child.out_features,
                    bias=child.bias is not None,
                    compute_dtype=torch.float16,
                    quant_type="nf4"
                )

                # Копируем веса и bias
                quantized_linear.weight.data = child.weight.data.clone()
                if child.bias is not None:
                    quantized_linear.bias.data = child.bias.data.clone()

                # Заменяем слой
                setattr(module, name, quantized_linear)
            else:
                quantize_linear_layers(child)

    quantize_linear_layers(quantized_draft)
    quantized_draft.to(device).eval()

    # Быстрая проверка
    print("Быстрая проверка работоспособности...")
    test_input = torch.tensor([[1, 2, 3, 4, 5]], device=device)

    with torch.no_grad():
        output = quantized_draft(test_input)
        next_token = torch.argmax(output[:, -1, :], dim=-1)

    print(f"  Модель работает. Следующий токен: {next_token.item()}")

    print("✅ Квантованная копия создана")
    return quantized_draft

In [19]:
def run_quantization_benchmark(draft_model, heavy_target):
    """
    Запускает бенчмарк с квантованной Draft моделью
    """
    print("\n" + "="*60)
    print("БЕНЧМАРК С КВАНТОВАНИЕМ DRAFT МОДЕЛИ")
    print("="*60)

    # Создаем квантованную копию
    quantized_draft = create_quantized_draft_from_existing(draft_model)

    # Используем существующий бенчмарк
    prompts = [
        "Write a Python function to calculate Fibonacci numbers.",
        "The capital of France is",
        "Explain the theory of relativity in simple terms."
    ]

    results = []
    for p in prompts:
        # Существующий бенчмарк
        # Std
        s_tok, s_time = autoregressive(heavy_target, p)
        s_speed = s_tok / s_time

        # Spec с оригинальной Draft
        start = time.time()
        _, stats_orig = speculative_sampling(p, 50, heavy_target, draft_model, tokenizer)
        spec_time_orig = time.time() - start
        spec_tok_orig = stats_orig['total_accepted'] + stats_orig['target_calls']
        spec_speed_orig = spec_tok_orig / spec_time_orig

        # Spec с квантованной Draft
        start = time.time()
        _, stats_quant = speculative_sampling(p, 50, heavy_target, quantized_draft, tokenizer)
        spec_time_quant = time.time() - start
        spec_tok_quant = stats_quant['total_accepted'] + stats_quant['target_calls']
        spec_speed_quant = spec_tok_quant / spec_time_quant

        results.append({
            "Prompt": p[:20],
            "Std (t/s)": f"{s_speed:.2f}",
            "Spec-Orig": f"{spec_speed_orig:.2f}",
            "Spec-Quant": f"{spec_speed_quant:.2f}",
            "Speedup-Orig": f"{spec_speed_orig/s_speed:.2f}x",
            "Speedup-Quant": f"{spec_speed_quant/s_speed:.2f}x"
        })

    # Вывод результатов
    print("\nРезультаты:")
    print(pd.DataFrame(results).to_string(float_format="{:.2f}".format, index=False))

    return quantized_draft, results

In [20]:
quantized_draft, results = run_quantization_benchmark(draft_model, heavy_target)


БЕНЧМАРК С КВАНТОВАНИЕМ DRAFT МОДЕЛИ
Создание квантованной копии Draft модели...
Быстрая проверка работоспособности...
  Модель работает. Следующий токен: 6
✅ Квантованная копия создана

Результаты:
              Prompt Std (t/s) Spec-Orig Spec-Quant Speedup-Orig Speedup-Quant
Write a Python funct     13.28     33.08      10.79        2.49x         0.81x
The capital of Franc     13.26     25.31       2.51        1.91x         0.19x
Explain the theory o     13.23     21.80      10.52        1.65x         0.80x


Что-то наоборот получилось замедление. Возможно, с игрушечными данными квантизация не так ускоряет, как в условиях продакшн.


### 2. LoRA Distillation (1 балл)
Представьте, что Draft модель плохо согласована с Target моделью. Попробуйте дообучить (Fine-Tune) Draft модель на наборе ответов Target модели, используя LoRA (Low-Rank Adaptation). Даже 500 примеров и 10-15 минут обучения на T4 могут повысить Acceptance Rate.

In [21]:
def generate_distillation_dataset(target_model, tokenizer, num_samples=500, max_length=256):
    """
    Генерирует датасет для дистилляции
    """
    print(f"Генерация {num_samples} примеров для дистилляции...")

    dataset_samples = []

    # Шаблоны для разнообразия
    prompt_templates = [
        "Explain {topic} in simple terms.",
        "What is {topic}?",
        "Describe {topic}.",
        "Tell me about {topic}.",
        "How does {topic} work?",
        "Why is {topic} important?",
        "The history of {topic}.",
        "Benefits of {topic}.",
        "Examples of {topic}.",
        "Difference between {topic} and {other}.",
        "Explain the concept of {topic}.",
        "What are the key features of {topic}?",
        "How is {topic} used in practice?",
        "What problems does {topic} solve?",
        "Future of {topic}.",
        "Applications of {topic}.",
        "Challenges in {topic}.",
        "Basic principles of {topic}.",
        "Advantages and disadvantages of {topic}.",
        "Step by step guide to {topic}."
    ]

    topics = [
        "quantum computing", "machine learning", "artificial intelligence",
        "blockchain", "climate change", "renewable energy", "neural networks",
        "Python programming", "deep learning", "computer vision",
        "natural language processing", "robotics", "space exploration",
        "genetic engineering", "cybersecurity", "electric vehicles",
        "virtual reality", "big data", "internet of things", "cloud computing",
        "algorithm", "data science", "encryption", "bioinformatics",
        "computer graphics", "distributed systems", "database management",
        "software engineering", "web development", "mobile applications",
        "game development", "operating systems", "computer networks",
        "human-computer interaction", "computational biology", "quantum physics",
        "mathematical optimization", "statistics", "probability theory",
        "linear algebra", "calculus", "discrete mathematics"
    ]

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        for i in tqdm(range(num_samples), desc="Generating samples"):
            # Случайный промпт (может повторяться)
            template = random.choice(prompt_templates)
            topic1 = random.choice(topics)

            if "{other}" in template:
                topic2 = random.choice([t for t in topics if t != topic1])
                prompt = template.format(topic=topic1, other=topic2)
            else:
                prompt = template.format(topic=topic1)

            # ИЛИ генерация полностью случайного промпта (30% случаев)
            if random.random() < 0.3:
                prompt_types = [
                    f"Write a short paragraph about {random.choice(topics)}.",
                    f"Explain {random.choice(topics)} to a beginner.",
                    f"What are the main components of {random.choice(topics)}?",
                    f"How would you teach {random.choice(topics)} to students?",
                    f"Compare {random.choice(topics)} and {random.choice(topics)}.",
                    f"Discuss the impact of {random.choice(topics)} on society.",
                    f"Provide a real-world example of {random.choice(topics)}.",
                    f"What are common misconceptions about {random.choice(topics)}?",
                    f"Summarize the key points about {random.choice(topics)}.",
                    f"Create a tutorial about {random.choice(topics)}."
                ]
                prompt = random.choice(prompt_types)

            # Токенизация и генерация
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

            # Генерация ответа
            target_response_ids = input_ids.clone()
            max_new_tokens = random.randint(32, 64)

            for _ in range(max_new_tokens):
                outputs = target_model(target_response_ids)
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                target_response_ids = torch.cat([target_response_ids, next_token], dim=1)

                if next_token.item() == tokenizer.eos_token_id:
                    break

            dataset_samples.append({
                'input_ids': input_ids[0].cpu(),
                'target_ids': target_response_ids[0].cpu()
            })

    print(f"✅ Сгенерировано {len(dataset_samples)} примеров")
    return dataset_samples

In [19]:
class DistillationDataset(Dataset):
    """Датасет для дистилляции"""
    def __init__(self, samples, tokenizer, max_seq_len=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Обрезаем если слишком длинная последовательность
        input_ids = sample['input_ids'][:self.max_seq_len]
        target_ids = sample['target_ids'][:self.max_seq_len]

        # Создаем маску внимания и labels для CE loss
        # Labels = target_ids смещенные на 1 вправо
        labels = target_ids.clone()
        labels[:-1] = target_ids[1:]
        labels[-1] = -100  # Игнорируем последний токен

        return {
            'input_ids': input_ids,
            'attention_mask': torch.ones_like(input_ids),
            'labels': labels
        }

def collate_fn(batch):
    """Collate function для DataLoader"""
    max_len = max(len(item['input_ids']) for item in batch)

    padded_batch = {
        'input_ids': torch.zeros(len(batch), max_len, dtype=torch.long),
        'attention_mask': torch.zeros(len(batch), max_len, dtype=torch.long),
        'labels': torch.full((len(batch), max_len), -100, dtype=torch.long)
    }

    for i, item in enumerate(batch):
        seq_len = len(item['input_ids'])
        padded_batch['input_ids'][i, :seq_len] = item['input_ids']
        padded_batch['attention_mask'][i, :seq_len] = item['attention_mask']
        padded_batch['labels'][i, :seq_len] = item['labels'][:seq_len]

    return padded_batch

In [23]:
# Генерация датасета
dataset_samples = generate_distillation_dataset(
    target_model, tokenizer, num_samples=500
)

Генерация 500 примеров для дистилляции...


Generating samples: 100%|██████████| 500/500 [14:29<00:00,  1.74s/it]

✅ Сгенерировано 500 примеров


In [ ]:
# Сохраняем
with open('distillation_dataset.pkl', 'wb') as f:
    pickle.dump(dataset_samples, f)
print(f"✅ Датсет сохранен: distillation_dataset.pkl ({len(dataset_samples)} примеров)")

In [15]:
with open('distillation_dataset.pkl', 'rb') as f:
    dataset_samples = pickle.load(f)
print(f"✅ Датсет загружен: {len(dataset_samples)} примеров")

✅ Датсет загружен: 500 примеров


In [24]:
batch_size = 16
dataset = DistillationDataset(dataset_samples, tokenizer)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

In [25]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module

        # Замораживаем базовые веса
        self.module.weight.requires_grad = False
        if self.module.bias is not None:
            self.module.bias.requires_grad = False

        #  LoRA адаптеры в том же типе, что и исходный слой
        weight_dtype = module.weight.dtype
        # LoRA адаптеры как параметры
        self.adapter_A = nn.Parameter(
            torch.empty(
                module.in_features,
                rank,
                device=module.weight.device,
                dtype=weight_dtype
            )
        )
        self.adapter_B = nn.Parameter(
            torch.zeros(
                rank,
                module.out_features,
                device=module.weight.device,
                dtype=weight_dtype
            )
        )

        # Инициализация
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        # adapter_B уже инициализирован нулями

    def forward(self, input):
        # Базовый forward
        original_output = self.module(input)

        # LoRA адаптация: input @ A @ B
        lora_output = input @ self.adapter_A @ self.adapter_B

        return original_output + lora_output

def add_lora_to_attention(attention_module, rank=8):
    """Добавляет LoRA адаптеры к Q,K,V,O проекциям в attention"""
    layers_to_lora = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

    for layer_name in layers_to_lora:
        if hasattr(attention_module, layer_name):
            base_layer = getattr(attention_module, layer_name)
            lora_layer = LoRALayer(base_layer, rank=rank)
            setattr(attention_module, layer_name, lora_layer)

def add_lora_to_draft_model(draft_model, rank=8):
    """Добавление LoRA адаптеров"""
    print("Добавление LoRA адаптеров к Draft модели...")

    # Копируем модель, чтобы не модифицировать оригинал
    import copy
    lora_model = copy.deepcopy(draft_model)
    # Заморозим всю модель
    for param in lora_model.parameters():
        param.requires_grad = False
    # Идем по всем слоям
    for name, module in lora_model.named_modules():
        if isinstance(module, NanoQwenBlock):
            # Заменяем слои
            module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=rank)
            module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=rank)
            module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=rank)
            module.self_attn.o_proj = LoRALayer(module.self_attn.o_proj, rank=rank)

            # Перемещаем на устройство (если нужно)
            module.self_attn.q_proj.to(device)
            module.self_attn.k_proj.to(device)
            module.self_attn.v_proj.to(device)
            module.self_attn.o_proj.to(device)

    # Считаем параметры
    trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in lora_model.parameters())

    print(f"  Trainable параметры: {trainable_params:,}")
    print(f"  Всего параметров: {total_params:,}")
    print(f"  % trainable: {100*trainable_params/total_params:.2f}%")

    return lora_model

In [41]:
def train_lora_distillation(draft_model, dataloader, num_epochs=5, lr=1e-4):
    """
    Обучение Draft модели с LoRA на ответах Target модели
    """
    print("\n" + "="*60)
    print("ОБУЧЕНИЕ DRAFT МОДЕЛИ С LoRA")
    print("="*60)

    # Добавляем LoRA адаптеры
    lora_draft = add_lora_to_draft_model(draft_model, rank=8)
    lora_draft_precise = lora_draft.to(torch.float32)
    lora_draft_precise.train()

    # Оптимизатор (только LoRA параметры)
    optimizer = torch.optim.AdamW(
        [p for p in lora_draft.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=0.0
    )

    # Обучение
    print(f"\nНачинаем обучение ({num_epochs} эпох)...")
    start_time = time.time()

    for epoch in range(num_epochs):
        total_loss = 0
        lora_draft.train()

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, batch in enumerate(progress_bar):
            # Перемещаем на GPU
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            # Forward Draft модели
            with torch.amp.autocast(device_type="cuda", enabled=False):
                outputs = lora_draft(input_ids)
                logits = outputs
                # CE loss (только для позиций с labels != -100)
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()

                loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    ignore_index=-100
                )
            if torch.isnan(loss) or torch.isinf(loss):
                print("⚠️ NaN/Inf loss detected!")
                # Не делаем backward, пропускаем batch
                optimizer.zero_grad()
                continue
            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in lora_draft.parameters() if p.requires_grad],
                max_norm=3
            )
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

    training_time = time.time() - start_time
    print(f"\n✅ Обучение завершено за {training_time/60:.1f} минут")

    # Переводим в eval режим
    trained_lora = lora_draft_precise.to(torch.float16)
    trained_lora.eval()

    return trained_lora

In [27]:
def test_acceptance_rate(draft_model, target_model, tokenizer, num_tests=20):
    """
    Тестирует Acceptance Rate после обучения
    """
    print("\n" + "="*60)
    print("ТЕСТИРОВАНИЕ ACCEPTANCE RATE")
    print("="*60)

    draft_model.eval()
    target_model.eval()

    total_accepted = 0
    total_drafted = 0

    test_prompts = [
        "Explain quantum computing.",
        "What is machine learning?",
        "Describe artificial intelligence.",
        "How does blockchain work?",
        "Tell me about climate change.",
        "What are neural networks?",
        "Explain Python programming.",
        "Describe deep learning.",
        "What is computer vision?",
        "How does NLP work?",
    ]

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        for prompt in tqdm(test_prompts[:num_tests], desc="Testing"):
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

            prefix_len = input_ids.shape[1]

            # Draft генерирует K=5 токенов
            draft_ids = input_ids.clone()
            for _ in range(5):
                outputs = draft_model(draft_ids)
                next_token = torch.argmax(outputs[:, -1, :], dim=-1, keepdim=True)
                draft_ids = torch.cat([draft_ids, next_token], dim=1)
                if next_token.item() == tokenizer.eos_token_id:
                    break

            drafted_tokens = draft_ids[0, prefix_len:]
            if len(drafted_tokens) == 0:
                continue

            # Target верифицирует
            target_outputs = target_model(draft_ids)
            target_logits = target_outputs.logits
            relevant_logits = target_logits[0, prefix_len-1:-1]
            target_preds = torch.argmax(relevant_logits, dim=-1)

            # Считаем совпадения
            accepted = 0
            for i in range(len(drafted_tokens)):
                if drafted_tokens[i] == target_preds[i]:
                    accepted += 1
                else:
                    break

            total_accepted += accepted
            total_drafted += len(drafted_tokens)

    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0
    print(f"\n📊 Результаты тестирования:")
    print(f"  Принято токенов: {total_accepted}/{total_drafted}")
    print(f"  Acceptance Rate: {acceptance_rate:.2%}")

    return acceptance_rate

In [43]:
def run_lora_distillation_experiment():
    """
    Эксперимент: LoRA, обучение, тестирование
    """
    print("🚀 ЗАПУСК ЭКСПЕРИМЕНТА LoRA DISTILLATION")
    print("="*60)

    # Дообучаем Draft модель
    trained_draft = train_lora_distillation(
        draft_model,
        dataloader,
        num_epochs=10,
        lr=1e-4
    )

    # Тестируем Acceptance Rate до обучения
    ar_before = test_acceptance_rate(
        draft_model,
        target_model,
        tokenizer,
        num_tests=20
    )

    # Тестируем Acceptance Rate после обучения
    ar_after = test_acceptance_rate(
        trained_draft,
        target_model,
        tokenizer,
        num_tests=20
    )

    # Выводим сравнение
    print(f"\n📊 СРАВНЕНИЕ ACCEPTANCE RATE:")
    print(f"  До обучения: {ar_before:.2%}")
    print(f"  После обучения: {ar_after:.2%}")
    print(f"  Улучшение: {((ar_after - ar_before) / ar_before * 100):+.1f}%")

    return trained_draft

In [44]:
trained_model = run_lora_distillation_experiment()

🚀 ЗАПУСК ЭКСПЕРИМЕНТА LoRA DISTILLATION

ОБУЧЕНИЕ DRAFT МОДЕЛИ С LoRA
Добавление LoRA адаптеров к Draft модели...
  Trainable параметры: 1,081,344
  Всего параметров: 631,248,768
  % trainable: 0.17%

Начинаем обучение (10 эпох)...


Epoch 1/10: 100%|██████████| 32/32 [00:04<00:00,  6.89it/s, loss=1.3173]


Epoch 1, Avg Loss: 3.8649


Epoch 2/10: 100%|██████████| 32/32 [00:04<00:00,  6.92it/s, loss=0.6889]


Epoch 2, Avg Loss: 0.9135


...

Epoch 10, Avg Loss: 0.4081

✅ Обучение завершено за 0.8 минут

ТЕСТИРОВАНИЕ ACCEPTANCE RATE


Testing: 100%|██████████| 10/10 [00:01<00:00,  6.72it/s]



📊 Результаты тестирования:
  Принято токенов: 39/50
  Acceptance Rate: 78.00%

ТЕСТИРОВАНИЕ ACCEPTANCE RATE


Testing: 100%|██████████| 10/10 [00:01<00:00,  5.49it/s]


📊 Результаты тестирования:
  Принято токенов: 29/50
  Acceptance Rate: 58.00%

📊 СРАВНЕНИЕ ACCEPTANCE RATE:
  До обучения: 78.00%
  После обучения: 58.00%
  Улучшение: -25.6%


Улучшить показатель так и не удалось, хотя много экспериментов было проведено. Пожалуй, lora тут только портит. Ну или мне не повезло в экспериментах.


### 3. Layer Pruning Distillation (1.5 балла)
Возьмите Target модель (1.5B) и создайте из нее Draft модель, просто удалив половину слоев (например, каждый второй). Получится "покалеченная" модель ~0.8B. Затем проведите Knowledge Distillation (обучите её восстанавливать логиты оригинальной модели) и используйте результат как Draft модель.

In [16]:
PRUNED_TARGET_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("Загрузка Target модели для pruning...")
pruned_target_config = AutoConfig.from_pretrained(PRUNED_TARGET_ID)

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(PRUNED_TARGET_ID)

# Загружаем модель целиком
pruned_target_model = AutoModelForCausalLM.from_pretrained(
    PRUNED_TARGET_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
pruned_target_model.eval()
print(f"Модель {PRUNED_TARGET_ID} загружена")
print(f"Количество слоев: {pruned_target_config.num_hidden_layers}")

Загрузка Target модели для pruning...
Загрузка токенизатора...
Модель Qwen/Qwen2.5-1.5B-Instruct загружена
Количество слоев: 28


In [17]:
def create_pruned_draft_from_target(target_model, target_config, keep_every_n=2):
    """
    Создает "покалеченную" Draft модель из Target модели,
    удаляя каждый N-ый слой (по умолчанию каждый второй)
    """
    print(f"\nСоздание pruned Draft модели (keep_every_n={keep_every_n})...")

    # 1. Создаем конфигурацию для Draft модели
    draft_config = copy.deepcopy(target_config)
    draft_config.num_hidden_layers = target_config.num_hidden_layers // keep_every_n

    print(f"  Оригинальных слоев: {target_config.num_hidden_layers}")
    print(f"  Остается слоев: {draft_config.num_hidden_layers}")

    # 2. Получаем веса Target модели
    target_state_dict = target_model.state_dict()

    # 3. Преобразуем ключи transformers -> наш формат
    transformed_target_state = {}
    for key, weight in target_state_dict.items():
        # Убираем префикс 'model.' если есть
        new_key = key
        if key.startswith('model.'):
            new_key = key[6:]  # Убираем 'model.'

        # Также преобразуем имена слоев если нужно
        # transformers: 'layers.0' -> наш формат тоже 'layers.0'
        transformed_target_state[new_key] = weight

    # 4. Теперь выбираем какие слои оставить
    layers_to_keep = list(range(0, target_config.num_hidden_layers, keep_every_n))
    print(f"  Сохраняем слои: {layers_to_keep}")

    # 5. Создаем пустую Draft модель
    pruned_draft = NanoQwen(draft_config)

    # 6. Собираем веса для Draft
    draft_state_dict = {}

    # Сначала копируем не-слоевые веса
    for key, weight in transformed_target_state.items():
        if 'layers' not in key:
            draft_state_dict[key] = weight

    # Затем копируем слои которые сохраняем
    for new_layer_idx, original_layer_idx in enumerate(layers_to_keep):
        for key, weight in transformed_target_state.items():
            if f'layers.{original_layer_idx}.' in key:
                # Заменяем номер слоя
                new_key = key.replace(
                    f'layers.{original_layer_idx}.',
                    f'layers.{new_layer_idx}.'
                )
                draft_state_dict[new_key] = weight

    print(f"\n  Перенесено ключей: {len(draft_state_dict)}/{len(transformed_target_state)}")

    # 7. Загружаем веса
    missing, unexpected = pruned_draft.load_state_dict(draft_state_dict, strict=False)

    print(f"  Загрузка весов:")
    print(f"    Отсутствующие ключи: {len(missing)}")
    print(f"    Неожиданные ключи: {len(unexpected)}")

    # 8. Проверяем критические missing
    critical_missing = [k for k in missing if 'layers' not in k]
    if critical_missing:
        print(f"  ⚠️  Критические missing (не слои): {critical_missing[:5]}")

    # 9. Перемещаем на устройство
    pruned_draft.to(device)
    pruned_draft.eval()

    # 10. Тест
    print(f"\n  Тестируем работоспособность...")
    test_input = torch.randint(0, 1000, (1, 16), device=device)
    with torch.no_grad():
        output = pruned_draft(test_input)
        next_token = torch.argmax(output[:, -1, :], dim=-1)

    print(f"    Forward успешен, next_token: {next_token.item()}")

    # 11. Параметры
    total_params = sum(p.numel() for p in pruned_draft.parameters())
    target_params = sum(p.numel() for p in target_model.parameters())

    print(f"\n  Параметры:")
    print(f"    Pruned Draft: {total_params:,}")
    print(f"    Original Target: {target_params:,}")
    print(f"    Сохранено: {100*total_params/target_params:.1f}%")

    return pruned_draft

# Создаем модель
pruned_draft_model = create_pruned_draft_from_target(
    pruned_target_model,
    pruned_target_config,
    keep_every_n=2
)


Создание pruned Draft модели (keep_every_n=2)...
  Оригинальных слоев: 28
  Остается слоев: 14
  Сохраняем слои: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26]

  Перенесено ключей: 171/339
  Загрузка весов:
    Отсутствующие ключи: 0
    Неожиданные ключи: 0

  Тестируем работоспособность...
    Forward успешен, next_token: 3512

  Параметры:
    Pruned Draft: 1,121,918,464
    Original Target: 1,543,714,304
    Сохранено: 72.7%


In [20]:
batch_size = 2
dataset = DistillationDataset(dataset_samples, tokenizer)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

In [21]:
def train_pruned_distillation(pruned_model, target_model, dataloader,
                             num_epochs=3, lr=1e-4):
    """
    Knowledge Distillation: обучаем pruned модель воспроизводить
    логиты оригинальной Target модели
    """
    print("\n" + "="*60)
    print("KNOWLEDGE DISTILLATION ДЛЯ PRUNED МОДЕЛИ")
    print("="*60)

    # 1. Подготовка модели
    pruned_model.train()
    pruned_model = pruned_model.to(torch.float32)  # Для стабильности

    optimizer = torch.optim.AdamW(
        pruned_model.parameters(),  # Обучаем ВСЮ pruned модель
        lr=lr,
        weight_decay=0.01
    )

    # 2. Обучение с MSE loss между логитами
    print(f"\nНачинаем обучение ({num_epochs} эпох)...")

    for epoch in range(num_epochs):
        total_loss = 0

        for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")):
            input_ids = batch['input_ids'].to(device)

            # Получаем логиты от Target модели
            with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                target_outputs = target_model(input_ids)
                target_logits = target_outputs.logits  # [batch, seq_len, vocab]

            # Forward pruned модели
            pruned_logits = pruned_model(input_ids)  # [batch, seq_len, vocab]

            # MSE loss между логитами (в float32 для стабильности)
            loss = F.mse_loss(
                pruned_logits.float(),
                target_logits.float()
            )

            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(pruned_model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            # Логирование каждые N батчей
            if batch_idx % 10 == 0:
                print(f"  Batch {batch_idx}: loss = {loss.item():.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

    # 3. Возвращаем в eval режим и float16
    pruned_model.eval()
    pruned_model = pruned_model.to(torch.float16)

    print(f"\n✅ Knowledge Distillation завершена")

    return pruned_model

# Используем существующий dataloader для обучения
distilled_pruned_model = train_pruned_distillation(
    pruned_draft_model,      # Наша pruned модель (0.8B)
    pruned_target_model,     # Оригинальная Target модель (1.5B)
    dataloader,              # Существующий DataLoader
    num_epochs=3,
    lr=1e-5
)


KNOWLEDGE DISTILLATION ДЛЯ PRUNED МОДЕЛИ

Начинаем обучение (3 эпох)...


Epoch 1/3:   0%|          | 1/250 [00:00<01:20,  3.07it/s]

  Batch 0: loss = 26.5230


Epoch 1/3:   4%|▍         | 11/250 [00:02<00:54,  4.42it/s]

  Batch 10: loss = 12.0295


...

  Batch 230: loss = 0.3298


Epoch 3/3:  96%|█████████▋| 241/250 [00:54<00:02,  4.48it/s]

  Batch 240: loss = 0.1940


Epoch 3/3: 100%|██████████| 250/250 [00:56<00:00,  4.46it/s]

Epoch 3, Avg Loss: 0.2533

✅ Knowledge Distillation завершена


In [22]:
def run_pruning_benchmark(original_draft, pruned_draft, heavy_target):
    """
    Бенчмарк обрезанной и исходной Draft моделей
    """
    print("\n" + "="*60)
    print("БЕНЧМАРК PRUNED DRAFT МОДЕЛИ")
    print("="*60)

    prompts = [
        "Write a Python function to calculate Fibonacci numbers.",
        "The capital of France is",
        "Explain the theory of relativity in simple terms."
    ]

    results = []
    for p in prompts:
        # Базовый метод (только Target)
        s_tok, s_time = autoregressive(heavy_target, p)
        s_speed = s_tok / s_time

        # Spec с оригинальной Draft
        start = time.time()
        _, stats_orig = speculative_sampling(p, 50, heavy_target, original_draft, tokenizer)
        spec_time_orig = time.time() - start
        spec_tok_orig = stats_orig['total_accepted'] + stats_orig['target_calls']
        spec_speed_orig = spec_tok_orig / spec_time_orig

        # Spec с обрезанной Draft
        start = time.time()
        _, stats_pruned = speculative_sampling(p, 50, heavy_target, pruned_draft, tokenizer)
        spec_time_pruned = time.time() - start
        spec_tok_pruned = stats_pruned['total_accepted'] + stats_pruned['target_calls']
        spec_speed_pruned = spec_tok_pruned / spec_time_pruned

        results.append({
            "Prompt": p[:20],
            "Std (t/s)": f"{s_speed:.2f}",
            "Spec-Orig": f"{spec_speed_orig:.2f}",
            "Spec-Pruned": f"{spec_speed_pruned:.2f}",
            "Speedup-Orig": f"{spec_speed_orig/s_speed:.2f}x",
            "Speedup-Pruned": f"{spec_speed_pruned/s_speed:.2f}x",
            "AR-Orig": f"{stats_orig['total_accepted']/stats_orig['total_drafted']:.2%}",
            "AR-Pruned": f"{stats_pruned['total_accepted']/stats_pruned['total_drafted']:.2%}"
        })

    # Вывод результатов
    print("\nРезультаты:")
    print(pd.DataFrame(results).to_string(index=False))

    return results

In [24]:
# Создаем HeavyTarget для бенчмарка
heavy_target = HeavyTarget(target_model, 0.04)

# Запускаем бенчмарк
pruning_results = run_pruning_benchmark(
    original_draft=draft_model,      # Исходная Draft модель 0.5B
    pruned_draft=distilled_pruned_model, # Обрезанная Draft модель
    heavy_target=heavy_target
)


БЕНЧМАРК PRUNED DRAFT МОДЕЛИ

Результаты:
              Prompt Std (t/s) Spec-Orig Spec-Pruned Speedup-Orig Speedup-Pruned AR-Orig AR-Pruned
Write a Python funct     13.02     31.73        9.09        2.44x          0.70x 100.00%     6.84%
The capital of Franc     13.15     24.53        6.91        1.87x          0.53x 100.00%     0.00%
Explain the theory o     12.99     21.09        9.61        1.62x          0.74x  60.00%     8.57%


In [25]:
print("\nПроверка forward pruned модели:")
test_input = torch.tensor([[1, 2, 3, 4, 5]], device=device)
with torch.no_grad():
    orig_out = draft_model(test_input)
    pruned_out = distilled_pruned_model(test_input)

print(f"Original output range: [{orig_out.min():.2f}, {orig_out.max():.2f}]")
print(f"Pruned output range: [{pruned_out.min():.2f}, {pruned_out.max():.2f}]")
print(f"Max difference: {torch.abs(orig_out - pruned_out).max():.4f}")


Проверка forward pruned модели:
Original output range: [-11.43, 18.62]
Pruned output range: [-14.35, 15.05]
Max difference: 19.7188


70% весов - слишком мало, чтоб научиться за три эпохи на 500 примерах хорошо имитировать большую модель. Поэтому получили такие плохие результаты. Принцип понятен.